### Notebook to create BLOCK-T413 for Camera Hexapod Optical State Estimation

This notebooks create the json block that goes one by one through all the Camera hexapod degrees of freedom and running the closed loop without applying corrections to check we are able to retrieve the optical state estimate.

Created on: 2025-03-24

Author: Guillem Megias

In [ ]:
from lsst.ts.observing import ObservingBlock, ObservingScript 
from lsst.ts.aos.analysis import build_configuration_schema
import os
import numpy as np

In [ ]:
current_path = os.getcwd()
block_number = 'T413'
name = 'BLOCK-T413'
program = name
reason = "WET-001"
note = "optical_state_cam_"
labels = ['dz', 'dx', 'dy', 'rx', 'ry']
constraints = []

### Define configuration schema

In [ ]:
# Define the configurable properties that we will use in the configuration schema
properties = {
    "filter": {
        "description": "Filter to use.",
        "type": "string",
        "default": "r_57"
    },
    "day": {
        "description": "Day of the year for the reference state.",
        "type": "integer",
        "default": 1
    },
    "seq": {
        "description": "Sequence number for the reference state.",
        "type": "integer",
        "default": 1
    },
    "exp_time": {
        "description": "Exposure time.",
        "type": "number",
        "default": 30.0
    },
    "maxiter": {
        "description": "Maximum number of iterations for the closed loop.",
        "type": "integer",
        "default": 1
    },
    "apply_corrections": {
        "description": "Apply corrections.",
        "type": "boolean",
        "default": False
    }
}

# Build the configuration schema for BLOCK-404
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

### Define scripts and block

In [ ]:
ranges = [150, 5000, 5000, 0.1, 0.1]
used_dofs = [[5, 6, 7], [6, 7], [6, 7], [8, 9], [8, 9]]
offset = 5
scripts = []

for idx, dof_range in enumerate(ranges):
    reset_state = ObservingScript(
        name="maintel/set_dof.py",
        standard=True,
        parameters= dict(
            day="$day",
            seqnum="$seqnum",
        )
    )

    dofs = np.zeros(50)
    dofs[idx + offset] = dof_range
    apply_state = ObservingScript(
        name="maintel/apply_dof.py",
        standard=True,
        parameters= dict(
            dofs=dofs.tolist(),
        )
    )

    closed_loop_script = ObservingScript(
        name="maintel/close_loop_lsstcam.py",
        standard=True,
        parameters= dict(
            filter="$filter",
            exp_time="$exp_time",
            max_iter="$maxiter",
            program="$program",
            reason=reason,
            note=f"{note}{labels[idx]}",
            used_dofs=used_dofs[idx],
            apply_corrections="$apply_corrections",
        )
    )
    scripts.append(reset_state)
    scripts.append(apply_state)
    scripts.append(closed_loop_script)

scripts.append(reset_state)

In [ ]:
block = ObservingBlock(
    name = name,
    program = program,
    configuration_schema=configuration_schema,
    scripts = scripts,
)

### Save configurable block

In [ ]:
block.model_dump_json(indent=2)

output_file_path = f'{current_path}/aos/ts_config_ocs/Scheduler/observing_blocks_maintel/AOS/WEP/{program}.json'

with open(output_file_path, 'w') as file:
    file.write(block.model_dump_json(indent=2))